# Agent Runner Smoke Test

Section-by-section notebook for testing `agent_runner.py` on a small sample (`n=5`) of SPL documents.

This notebook is organized to inspect:
1. sample SPL selection
2. DailyMed contraindications extraction
3. contraindication item extraction
4. candidate retrieval + prefiltering
5. direct verification + route-or-fill
6. full pipeline run
7. aggregation
8. optional evaluation

In [2]:
!nvidia-smi

Fri Mar 20 14:43:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.51.03              Driver Version: 575.51.03      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:27:00.0 Off |                    0 |
| N/A   25C    P0             59W /  400W |      92MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 0. Setup

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

from agent_runner import (
    AGG_CSV_COLUMNS,
    AgentRunConfig,
    AzureChatLLM,
    AzureOpenAIConfig,
    ContraAgent,
    aggregate_agent_results,
    evaluate_aggregated_predictions,
    extract_items_for_spl,
    load_spl_records_from_file,
    prefilter_slot_candidates,
    retrieve_candidates_for_item,
    write_csv_rows,
    write_jsonl,
)
from VaxMapper.src.utils.dailymed import CONTRA_Loinc, extract_section

pd.set_option("display.max_colwidth", 200)
ROOT = Path.cwd()
ROOT

PosixPath('/data/wmanuel3/VaxMapperRepo')

## 1. Configuration

This notebook uses the Azure OpenAI SDK through `agent_runner.AzureChatLLM`.

In [ ]:
SPL_LIST_PATH = None
FALLBACK_GOLD_CSV = ROOT / "results/contra_gold_100_1.csv"
SAMPLE_N = 5
OUTPUT_DIR = ROOT / "results/agent_runner_smoketest"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_JSONL = OUTPUT_DIR / "agent_results.jsonl"
AGG_JSONL = OUTPUT_DIR / "aggregated_hits.jsonl"
AGG_CSV = OUTPUT_DIR / "aggregated_hits.csv"
EVAL_JSON = OUTPUT_DIR / "eval_metrics.json"
EVAL_DETAILS_CSV = OUTPUT_DIR / "eval_details.csv"

# required_env = [
#     "AZURE_OPENAI_ENDPOINT",
#     "AZURE_OPENAI_API_KEY",
#     "AZURE_OPENAI_DEPLOYMENT",
#     "AZURE_OPENAI_API_VERSION",
# ]
# missing_env = [key for key in required_env if not os.environ.get(key)]
# if missing_env:
#     raise RuntimeError(f"Missing Azure OpenAI env vars: {missing_env}")




cfg = AzureOpenAIConfig(
    endpoint=endpoint,
    api_key=subscription_key,
    deployment=deployment,
    api_version=api_version,
    # max_retries=2,
)
llm = AzureChatLLM(cfg)
agent = ContraAgent(llm, AgentRunConfig())

OUTPUT_DIR

PosixPath('/data/wmanuel3/VaxMapperRepo/results/agent_runner_smoketest')

## 2. Load a Sample of SPL_SET_ID Values

In [13]:
if SPL_LIST_PATH:
    spl_records = load_spl_records_from_file(str(SPL_LIST_PATH))[:SAMPLE_N]
else:
    gold_df = pd.read_csv(FALLBACK_GOLD_CSV)
    spl_ids = gold_df["SPL_SET_ID"].dropna().astype(str).drop_duplicates().head(SAMPLE_N).tolist()
    spl_records = [{"SPL_SET_ID": spl_id} for spl_id in spl_ids]

sample_ids = [row["SPL_SET_ID"] for row in spl_records]
sample_ids

['601f7353-225d-4a5d-9810-3869c2c21874',
 '51720017-e33f-4e3c-8986-a68291ae4879',
 'a7701e8c-4e16-4165-8b83-de2c08d5ded3',
 '37b5963b-0bbe-073e-e054-00144ff8d46c',
 '6d04b514-c85c-4db1-919e-21dd98172791']

## 3. DailyMed Fetch: Contraindications Section + Product Name

In [19]:
spl_contexts = []
for spl_id in sample_ids:
    payload = extract_section(str(spl_id), [CONTRA_Loinc])
    section_payload = (payload.get("sections") or {}).get(CONTRA_Loinc, {})
    spl_contexts.append(
        {
            "SPL_SET_ID": str(spl_id),
            "product_name": payload.get("product_name"),
            "contra_section_found": bool(section_payload.get("section_text")),
            "contra_section_text": section_payload.get("section_text") or "",
        }
    )

display(pd.DataFrame(spl_contexts)[["SPL_SET_ID", "product_name", "contra_section_found"]])

,SPL_SET_ID,product_name,contra_section_found
0,601f7353-225d-4a5d-9810-3869c2c21874,Phenytoin Sodium,True
1,51720017-e33f-4e3c-8986-a68291ae4879,BUPROPION HYDROCHLORIDE (SR),True
2,a7701e8c-4e16-4165-8b83-de2c08d5ded3,Diclofenac sodium and Misoprostol,True
3,37b5963b-0bbe-073e-e054-00144ff8d46c,phendimetrazine tartrate,True
4,6d04b514-c85c-4db1-919e-21dd98172791,Paclitaxel,True


## 4. Step 1 Preview: Contraindication Item Extraction

In [5]:
extracted_preview = []
for ctx in spl_contexts:
    items = extract_items_for_spl(
        ctx,
        chat_fn=llm.chat,
        max_tokens=agent.cfg.max_llm_tokens_mid,
        stop=agent.cfg.stop,
        retries=agent.cfg.retries,
    )
    for idx, item in enumerate(items):
        extracted_preview.append(
            {
                "SPL_SET_ID": ctx["SPL_SET_ID"],
                "product_name": ctx.get("product_name"),
                "item_index": idx,
                **item,
            }
        )

extracted_df = pd.DataFrame(extracted_preview)

In [6]:
extracted_df

,SPL_SET_ID,product_name,item_index,ci_text,contraindication_state_text,substance_text,severity_span,course_span
0,601f7353-225d-4a5d-9810-3869c2c21874,Phenytoin Sodium,0,A history of hypersensitivity to phenytoin,hypersensitivity,phenytoin,None,history
1,601f7353-225d-4a5d-9810-3869c2c21874,Phenytoin Sodium,1,A history of hypersensitivity to its inactive ingredients,hypersensitivity,its inactive ingredients,None,history
2,601f7353-225d-4a5d-9810-3869c2c21874,Phenytoin Sodium,2,A history of hypersensitivity to other hydantoins,hypersensitivity,other hydantoins,None,history
3,601f7353-225d-4a5d-9810-3869c2c21874,Phenytoin Sodium,3,A history of prior acute hepatotoxicity attributable to phenytoin,hepatotoxicity,phenytoin,acute,prior
4,601f7353-225d-4a5d-9810-3869c2c21874,Phenytoin Sodium,4,Coadministration with delavirdine,concomitant administration,delavirdine,None,None


## 5. Step 2 + 4 Preview: Retrieval and Prefiltering on One Item

In [7]:
preview_item = extracted_preview[0].copy()
raw_candidates = retrieve_candidates_for_item(preview_item)
prefiltered_candidates = prefilter_slot_candidates(raw_candidates)

candidate_summary = {
    "ci_text": preview_item.get("ci_text"),
    "focus_candidates": len(raw_candidates.get("focus_candidates", [])),
    "direct_candidates": len(raw_candidates.get("direct_candidates", [])),
    "causative_agent_candidates": len(raw_candidates.get("causative_agent_candidates", [])),
    "severity_candidates": len(raw_candidates.get("severity_candidates", [])),
    "clinical_course_candidates": len(raw_candidates.get("clinical_course_candidates", [])),
    "causative_agent_candidates_prefiltered": len(prefiltered_candidates.get("causative_agent_candidates", [])),
    "severity_candidates_prefiltered": len(prefiltered_candidates.get("severity_candidates", [])),
    "clinical_course_candidates_prefiltered": len(prefiltered_candidates.get("clinical_course_candidates", [])),
}
candidate_summary

Deleting existing index: snomed_ct_es_index
Creating index 'snomed_ct_es_index'...
Loading model tavakolih/all-MiniLM-L6-v2-pubmed-full...
Model tavakolih/all-MiniLM-L6-v2-pubmed-full loaded to cuda:0 successfully.
Moving FAISS index to GPU (device 0)...
Index moved to GPU successfully.


{'ci_text': 'A history of hypersensitivity to phenytoin',
 'focus_candidates': 20,
 'direct_candidates': 20,
 'causative_agent_candidates': 20,
 'severity_candidates': 0,
 'clinical_course_candidates': 20,
 'causative_agent_candidates_prefiltered': 7,
 'severity_candidates_prefiltered': 0,
 'clinical_course_candidates_prefiltered': 0}

In [9]:
display(pd.DataFrame(prefiltered_candidates.get("focus_candidates", [])[:10]))
display(pd.DataFrame(prefiltered_candidates.get("causative_agent_candidates", [])[:10]))
display(pd.DataFrame(prefiltered_candidates.get("severity_candidates", [])[:10]))
display(pd.DataFrame(prefiltered_candidates.get("clinical_course_candidates", [])[:10]))

,id,label,fused,from
0,252097006,Hypersensitivity finding (finding),0.032522,"bm25,dense"
1,473010000,Hypersensitivity condition (finding),0.029911,"bm25,dense"
2,609433001,Hypersensitivity disposition (finding),0.029469,"bm25,dense"
3,472963003,Hypersensitivity process (qualifier value),0.029387,"bm25,dense"
4,421961002,Hypersensitivity reaction (disorder),0.028986,"bm25,dense"
5,272401006,Hypersensitivity types (qualifier value),0.028405,"bm25,dense"
6,21626009,Cutaneous hypersensitivity (disorder),0.026667,"bm25,dense"
7,609405001,Non-allergic hypersensitivity condition (finding),0.026387,"bm25,dense"
8,609396006,Non-allergic hypersensitivity disposition (finding),0.026289,"bm25,dense"
9,29268000,Contact hypersensitivity (disorder),0.025942,"bm25,dense"


,id,label,fused,from
0,387220006,Phenytoin (substance),0.031778,"bm25,dense"
1,706933005,Free phenytoin (substance),0.031281,"bm25,dense"
2,387416005,Phenytoin sodium (substance),0.028694,"bm25,dense"
3,445821000221103,Phenytoin calcium (substance),0.028083,"bm25,dense"
4,777187002,Product containing only phenytoin (medicinal product),0.027433,"bm25,dense"
5,40556005,Product containing phenytoin (medicinal product),0.026838,"bm25,dense"
6,322944003,Product containing precisely phenytoin sodium 50 milligram/1 each conventional release oral capsule (clinical drug),0.024607,"bm25,dense"


""


""


## 6. Step 3 + 5 Preview: Direct Verify and Minimal Representation on One Item

In [10]:
preview_item

{'SPL_SET_ID': '601f7353-225d-4a5d-9810-3869c2c21874',
 'product_name': 'Phenytoin Sodium',
 'item_index': 0,
 'ci_text': 'A history of hypersensitivity to phenytoin',
 'contraindication_state_text': 'hypersensitivity',
 'substance_text': 'phenytoin',
 'severity_span': None,
 'course_span': 'history'}

In [11]:
direct_preview = agent.verify_direct_match(
    preview_item["ci_text"],
    prefiltered_candidates.get("direct_candidates", prefiltered_candidates.get("focus_candidates", [])),
)

route_or_fill_preview = None
if direct_preview.get("direct_match") is not True:
    route_or_fill_preview = agent.route_or_fill(preview_item, prefiltered_candidates)

print("DIRECT VERIFY")
print(json.dumps({k: v for k, v in direct_preview.items() if k != "raw"}, indent=2))
print()
print("ROUTE OR FILL / MINIMAL REPRESENTATION")
print(json.dumps({} if route_or_fill_preview is None else {k: v for k, v in route_or_fill_preview.items() if k != "raw"}, indent=2))

DIRECT VERIFY
{
  "direct_match": false,
  "selected_id": "N/A",
  "selected_term": "N/A"
}

ROUTE OR FILL / MINIMAL REPRESENTATION
{
  "post_decision": "N/A",
  "selected_focus_id": "252097006",
  "fills": {
    "causative_agent": "387220006",
    "severity": "N/A",
    "clinical_course": "N/A"
  }
}


## 7. Full Sample Run (`n=5`)

In [16]:
run_rows = []
for ctx in spl_contexts:
    run_rows.append(agent.process_spl(ctx))

write_jsonl(str(RAW_JSONL), run_rows)
print(f"Wrote raw results to: {RAW_JSONL}")

flat_results = []
for row in run_rows:
    for item in row.get("results") or []:
        flat_results.append(
            {
                "SPL_SET_ID": row.get("SPL_SET_ID"),
                "product_name": row.get("product_name"),
                **item,
            }
        )

flat_df = pd.DataFrame(flat_results)
preview_columns = [
    "SPL_SET_ID",
    "item_index",
    "status",
    "post_decision",
    "query_text",
    "selected_id",
    "selected_term",
    "selected_focus_id",
    "selected_focus_term",
    "expression",
]
display(flat_df.reindex(columns=preview_columns).head(50))

Wrote raw results to: /data/wmanuel3/VaxMapperRepo/results/agent_runner_smoketest/agent_results.jsonl


,SPL_SET_ID,item_index,status,post_decision,query_text,selected_id,selected_term,selected_focus_id,selected_focus_term,expression
0,601f7353-225d-4a5d-9810-3869c2c21874,0,MINIMAL,N/A,A history of hypersensitivity to phenytoin,NaN,NaN,252097006,Hypersensitivity finding (finding),252097006:{246075003=387220006}
1,601f7353-225d-4a5d-9810-3869c2c21874,1,MINIMAL,N/A,A history of hypersensitivity to its inactive ingredients,NaN,NaN,252097006,Hypersensitivity finding (finding),252097006:{246075003=27566006}
2,601f7353-225d-4a5d-9810-3869c2c21874,2,MINIMAL,N/A,A history of hypersensitivity to other hydantoins,NaN,NaN,473010000,Hypersensitivity condition (finding),473010000:{246075003=871696008}
3,601f7353-225d-4a5d-9810-3869c2c21874,3,MINIMAL,N/A,A history of prior acute hepatotoxicity attributable to phenytoin,NaN,NaN,427399008,Drug-induced disorder of liver (disorder),427399008:{246075003=387220006}
4,601f7353-225d-4a5d-9810-3869c2c21874,4,DIRECT,NaN,Coadministration with delavirdine,108710001,Delavirdine (substance),NaN,NaN,NaN
5,a7701e8c-4e16-4165-8b83-de2c08d5ded3,0,DIRECT,NaN,Pregnancy,77386006,Pregnancy (finding),NaN,NaN,NaN
6,a7701e8c-4e16-4165-8b83-de2c08d5ded3,1,MINIMAL,N/A,In the setting of coronary artery bypass graft (CABG) surgery,NaN,NaN,232717009,Coronary artery bypass grafting (procedure),232717009
7,a7701e8c-4e16-4165-8b83-de2c08d5ded3,2,MINIMAL,N/A,Active gastrointestinal bleeding,NaN,NaN,74474003,Gastrointestinal hemorrhage (disorder),74474003
8,a7701e8c-4e16-4165-8b83-de2c08d5ded3,3,MINIMAL,N/A,History of asthma after taking aspirin,NaN,NaN,281239006,Exacerbation of asthma (disorder),281239006:{246075003=387458008}
9,a7701e8c-4e16-4165-8b83-de2c08d5ded3,4,MINIMAL,N/A,History of asthma after taking other NSAIDs,NaN,NaN,281239006,Exacerbation of asthma (disorder),281239006:{246075003=372665008}


## 8. Aggregation Outputs

In [17]:
aggregated_rows, csv_rows = aggregate_agent_results(run_rows)
write_jsonl(str(AGG_JSONL), aggregated_rows)
write_csv_rows(str(AGG_CSV), csv_rows, AGG_CSV_COLUMNS)

print(f"Wrote aggregated JSONL to: {AGG_JSONL}")
print(f"Wrote aggregated CSV to: {AGG_CSV}")
display(pd.DataFrame(csv_rows).head(50))

Wrote aggregated JSONL to: /data/wmanuel3/VaxMapperRepo/results/agent_runner_smoketest/aggregated_hits.jsonl
Wrote aggregated CSV to: /data/wmanuel3/VaxMapperRepo/results/agent_runner_smoketest/aggregated_hits.csv


,SPL_SET_ID,item_index,query_text,mapping_source,final_concept_id,final_concept_term,postcoord_expression,causative_agent_id,causative_agent_term,severity_id,severity_term,clinical_course_id,clinical_course_term
0,601f7353-225d-4a5d-9810-3869c2c21874,0,A history of hypersensitivity to phenytoin,minimal,252097006,Hypersensitivity finding (finding),252097006:{246075003=387220006},387220006,Phenytoin (substance),,,,
1,601f7353-225d-4a5d-9810-3869c2c21874,1,A history of hypersensitivity to its inactive ingredients,minimal,252097006,Hypersensitivity finding (finding),252097006:{246075003=27566006},27566006,Drug preservative (product),,,,
2,601f7353-225d-4a5d-9810-3869c2c21874,2,A history of hypersensitivity to other hydantoins,minimal,473010000,Hypersensitivity condition (finding),473010000:{246075003=871696008},871696008,Hydantoin and/or hydantoin derivative (substance),,,,
3,601f7353-225d-4a5d-9810-3869c2c21874,3,A history of prior acute hepatotoxicity attributable to phenytoin,minimal,427399008,Drug-induced disorder of liver (disorder),427399008:{246075003=387220006},387220006,Phenytoin (substance),,,,
4,601f7353-225d-4a5d-9810-3869c2c21874,4,Coadministration with delavirdine,verified,108710001,Delavirdine (substance),108710001,,,,,,
5,6d04b514-c85c-4db1-919e-21dd98172791,0,history of hypersensitivity reactions to paclitaxel,minimal,421961002,Hypersensitivity reaction (disorder),421961002:{246075003=387374002},387374002,Paclitaxel (substance),,,,
6,6d04b514-c85c-4db1-919e-21dd98172791,1,history of hypersensitivity reactions to other drugs formulated in polyoxyl 35 castor oil,minimal,421961002,Hypersensitivity reaction (disorder),421961002:{246075003=34239008},34239008,Castor oil (substance),,,,
7,6d04b514-c85c-4db1-919e-21dd98172791,2,"patients with solid tumors who have baseline neutrophil counts of <1,500 cells/mm3",minimal,165517008,Neutropenia (finding),165517008,,,,,,
8,6d04b514-c85c-4db1-919e-21dd98172791,3,"patients with AIDS-related Kaposi's sarcoma with baseline neutrophil counts of <1,000 cells/mm3",minimal,165517008,Neutropenia (finding),165517008,,,,,,
9,a7701e8c-4e16-4165-8b83-de2c08d5ded3,0,Pregnancy,verified,77386006,Pregnancy (finding),77386006,,,,,,


## 9. Optional Evaluation

In [18]:
RUN_EVAL = True
GOLD_CSV = FALLBACK_GOLD_CSV

if RUN_EVAL:
    metrics = evaluate_aggregated_predictions(
        pred_csv=str(AGG_CSV),
        gold_csv=str(GOLD_CSV),
        out_json=str(EVAL_JSON),
        out_details_csv=str(EVAL_DETAILS_CSV),
    )
    display(metrics)
    print(f"Wrote eval JSON to: {EVAL_JSON}")
    print(f"Wrote eval details CSV to: {EVAL_DETAILS_CSV}")
else:
    print("Skipping evaluation.")

Loading model tavakolih/all-MiniLM-L6-v2-pubmed-full...
Model tavakolih/all-MiniLM-L6-v2-pubmed-full loaded to cuda:0 successfully.
{
  "inputs": {
    "pred_csv": "/data/wmanuel3/VaxMapperRepo/results/agent_runner_smoketest/aggregated_hits.csv",
    "gold_csv": "/data/wmanuel3/VaxMapperRepo/results/contra_gold_100_1.csv",
    "gold_rows_total": 387,
    "gold_rows_used": 314,
    "concept_gold_rows_used": 163,
    "gold_rows_dropped_na_expression": 73,
    "discard_na_gold_expression": true,
    "pred_rows": 22,
    "gold_unique_spl": 100,
    "semantic_matching_enabled": true,
    "semantic_matching_unavailable_reasons": [],
    "alpha": 0.85,
    "beta": 0.15,
    "min_pair_score": 0.0
  },
  "concept_level": {
    "tp": 2,
    "fp": 8,
    "fn": 161,
    "precision": 0.2,
    "recall": 0.012269938650306749,
    "f1": 0.023121387283236993
  },
  "attribute_level": {
    "tp": 22,
    "fp": 11,
    "fn": 469,
    "precision": 0.6666666666666666,
    "recall": 0.04480651731160896,
   

{'inputs': {'pred_csv': '/data/wmanuel3/VaxMapperRepo/results/agent_runner_smoketest/aggregated_hits.csv',
  'gold_csv': '/data/wmanuel3/VaxMapperRepo/results/contra_gold_100_1.csv',
  'gold_rows_total': 387,
  'gold_rows_used': 314,
  'concept_gold_rows_used': 163,
  'gold_rows_dropped_na_expression': 73,
  'discard_na_gold_expression': True,
  'pred_rows': 22,
  'gold_unique_spl': 100,
  'semantic_matching_enabled': True,
  'semantic_matching_unavailable_reasons': [],
  'alpha': 0.85,
  'beta': 0.15,
  'min_pair_score': 0.0},
 'concept_level': {'tp': 2,
  'fp': 8,
  'fn': 161,
  'precision': 0.2,
  'recall': 0.012269938650306749,
  'f1': 0.023121387283236993},
 'attribute_level': {'tp': 22,
  'fp': 11,
  'fn': 469,
  'precision': 0.6666666666666666,
  'recall': 0.04480651731160896,
  'f1': 0.08396946564885495}}

Wrote eval JSON to: /data/wmanuel3/VaxMapperRepo/results/agent_runner_smoketest/eval_metrics.json
Wrote eval details CSV to: /data/wmanuel3/VaxMapperRepo/results/agent_runner_smoketest/eval_details.csv
